# P1 - Ciência de Dados

Em Ciência de Dados, antes de qualquer análise ou modelo, é preciso entender o problema de negócio, avaliar a qualidade dos dados e prepará-los. Dados brutos quase sempre chegam sujos: tipos errados, valores ausentes, duplicatas, categorias escritas de formas diferentes e valores impossíveis. Nesta prova você vai diagnosticar uma planilha bruta, limpar e transformar os dados com Python / Pandas, aplicar estatística descritiva e responder perguntas de negócio usando probabilidade, considerando a loja online TechShop.

O conteúdo cobrado vai até a Aula 04: Ciência de Dados & Big Data e ciclo de vida do dado (Aula 01), Python para Ciência de Dados (Aula 02), Estatística Descritiva Aplicada (Aula 03) e Probabilidade Aplicada a Negócios (Aula 04).

## Importações


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

## Contexto do Negócio — Loja Online TechShop

A TechShop vende eletrônicos e itens de informática pela internet. A equipe exportou de uma planilha o histórico recente de pedidos para começar a análise de vendas. O arquivo veio desorganizado: mesma informação escrita de jeitos diferentes, números como texto, datas em formatos distintos, linhas repetidas e valores que não fazem sentido. Você foi encarregado(a) de preparar esses dados e extrair os primeiros indicadores.

## Questões

### Questão 1 (1,5 ponto) — Diagnóstico dos Dados e Enquadramento em Ciência de Dados

Analisando o Registro Bruto da TechShop, responda em células markdown ou como comentários:

**a)** (0,5) Liste todos os problemas de qualidade de dados que você identifica na planilha (tipos incorretos, valores ausentes, duplicatas, categorias inconsistentes, formatos de data, valores impossíveis). Relacione com o princípio GIGO.

* Dados duplicados, 2 pedidos com o mesmo id 1003 (não pode ter mais de um pedido com a mesma numeração).
* Diferente formatação em cliente, uf, categoria, produto e canal.
* Tipo incorreto e formatação incorreta em preco_unitario, precisa ser float.
* Valores ausentes e impossíveis em quantidade.

    Todas as informações devem seguir um padrão determinado para depois se realizar o processamento desses dados.

**b)** (0,5) Classifique cada uma das 11 colunas quanto ao tipo de variável (qualitativa nominal, qualitativa ordinal, quantitativa discreta, quantitativa contínua ou temporal). Justifique as duas que você considerou mais difíceis.

* id_pedido - qualitativa ordinal.
* data - quantitativa temporal.
* cliente - qualitativa nominal.
* uf - qualitativa nominal.
* categoria - qualitativa nominal.
* produto - qualitativa nominal.
* preco_unitario - quantitativa contínua.
* quantidade - quantitativa contínua.
* canal - qualitativa nominal.
* avaliacao - quantitativa contínua.
* status - qualitativa nominal.

**c)** (0,5) Indique em que fase do ciclo de vida do dado / CRISP-DM essas tarefas se encontram e escreva o seu plano de limpeza (passo a passo), citando a Documentação de Apoio como referência de método.

* Fase de tratamento de dados.

* Plano:
    1. Remover duplicatas.
    2. Converter data para datetime.
    3. Padronizar textos: strip + Title Case + mapa de valores canônicos.
    4. Converter preco_unitario para float (tirar "R$", milhar e trocar "," por ".").
    5. Tratar valores impossíveis: duracao_min <= 0 vira ausente (NaN).
    6. Decidir o que fazer com ausentes (imputar pela mediana ou manter NaN).

### Questão 2 (1,5 ponto) — Carga e Limpeza dos Dados

Refatorando o método da Documentação de Apoio para a TechShop:

**a)** (0,3) Carregue o Registro Bruto em um DataFrame (reconstruindo a lista de dicionários ou lendo um .csv) e mostre info(), isnull().sum() e duplicated().sum().

In [2]:
bruto = [
    {
        "id_pedido": 1001,
        "data": "2025-01-05",
        "cliente": "Ana Beatriz Lima",
        "uf": "RJ",
        "categoria": "Eletrônicos",
        "produto": "Fone Bluetooth",
        "preco_unitario": "199,90",
        "quantidade": 2,
        "canal": "App",
        "avaliacao": 5,
        "status": "Entregue"
    },
    {
        "id_pedido": 1002,
        "data": "06/01/2025",
        "cliente": "João P. Souza",
        "uf": "rj",
        "categoria": "eletronicos",
        "produto": "Carregador USB-C",
        "preco_unitario": "R$ 89,90",
        "quantidade": 1,
        "canal": "Site",
        "avaliacao": 4,
        "status": "entregue"
    },
    {
        "id_pedido": 1003,
        "data": "2025-01-07",
        "cliente": "MARINA ROCHA",
        "uf": "SP",
        "categoria": "Informática",
        "produto": "Mouse sem fio",
        "preco_unitario": "129,90",
        "quantidade": 3,
        "canal": "app",
        "avaliacao": 3,
        "status": "Entregue"
    },
    {
        "id_pedido": 1003,
        "data": "2025-01-07",
        "cliente": "MARINA ROCHA",
        "uf": "SP",
        "categoria": "Informática",
        "produto": "Mouse sem fio",
        "preco_unitario": "129,90",
        "quantidade": 3,
        "canal": "app",
        "avaliacao": 3,
        "status": "Entregue"
    },
    {
        "id_pedido": 1004,
        "data": "08/01/2025",
        "cliente": "Carlos Andrade",
        "uf": "São Paulo",
        "categoria": "Informática",
        "produto": "Teclado mecânico",
        "preco_unitario": "349,00",
        "quantidade": 1,
        "canal": "Site",
        "avaliacao": 5,
        "status": "Entregue"
    },
    {
        "id_pedido": 1005,
        "data": "2025-01-09",
        "cliente": "ana beatriz lima",
        "uf": "RJ",
        "categoria": "Acessórios",
        "produto": "Capa de notebook",
        "preco_unitario": "79,90",
        "quantidade": 2,
        "canal": "Loja física",
        "avaliacao": None,
        "status": "Pendente"
    },
    {
        "id_pedido": 1006,
        "data": "2025-01-10",
        "cliente": "Pedro Henrique",
        "uf": "MG",
        "categoria": "Eletrônicos",
        "produto": "Caixa de som",
        "preco_unitario": "259,90",
        "quantidade": 1,
        "canal": "App",
        "avaliacao": 4,
        "status": "Entregue"
    },
    {
        "id_pedido": 1007,
        "data": "10/01/2025",
        "cliente": "Juliana Martins",
        "uf": "mg",
        "categoria": "ELETRÔNICOS",
        "produto": "Smartband",
        "preco_unitario": "199,00",
        "quantidade": -1,
        "canal": "Site",
        "avaliacao": 2,
        "status": "Cancelado"
    },
    {
        "id_pedido": 1008,
        "data": "2025-01-11",
        "cliente": "Rafael Gomes",
        "uf": "RJ",
        "categoria": "Informática",
        "produto": "SSD 480GB",
        "preco_unitario": "329,90",
        "quantidade": 0,
        "canal": "App",
        "avaliacao": None,
        "status": "Cancelado"
    },
    {
        "id_pedido": 1009,
        "data": "12/01/2025",
        "cliente": "Beatriz Nunes",
        "uf": "BA",
        "categoria": "Acessórios",
        "produto": "Suporte para celular",
        "preco_unitario": "39,90",
        "quantidade": 4,
        "canal": "site",
        "avaliacao": 5,
        "status": "Entregue"
    },
    {
        "id_pedido": 1010,
        "data": "2025-01-12",
        "cliente": "Tiago Ferreira",
        "uf": "Bahia",
        "categoria": "Acessórios",
        "produto": "Cabo HDMI",
        "preco_unitario": "49,90",
        "quantidade": None,
        "canal": "App",
        "avaliacao": 3,
        "status": "Entregue"
    },
    {
        "id_pedido": 1011,
        "data": "13/01/2025",
        "cliente": "Larissa Dias",
        "uf": "RJ",
        "categoria": "Eletrônicos",
        "produto": "Headphone Premium",
        "preco_unitario": "1.199,90",
        "quantidade": 1,
        "canal": "Site",
        "avaliacao": 1,
        "status": "Entregue"
    },
    {
        "id_pedido": 1012,
        "data": "2025-01-14",
        "cliente": "GUSTAVO LIMA",
        "uf": "pr",
        "categoria": "informática",
        "produto": "Webcam HD",
        "preco_unitario": "159,90",
        "quantidade": 2,
        "canal": "app",
        "avaliacao": 4,
        "status": "Entregue"
    },
    {
        "id_pedido": 1013,
        "data": "14/01/2025",
        "cliente": "Camila Souza",
        "uf": "PR",
        "categoria": "Acessórios",
        "produto": "Mousepad",
        "preco_unitario": "29,90",
        "quantidade": 5,
        "canal": "Loja Física",
        "avaliacao": 5,
        "status": "Entregue"
    },
    {
        "id_pedido": 1014,
        "data": "2025-01-15",
        "cliente": "Anderson Melo",
        "uf": "RJ",
        "categoria": "Eletrônicos",
        "produto": "Smart TV 50\"",
        "preco_unitario": "2.499,00",
        "quantidade": 1,
        "canal": "Site",
        "avaliacao": 4,
        "status": "Entregue"
    },
    {
        "id_pedido": 1015,
        "data": "15/01/2025",
        "cliente": "Fernanda Castro",
        "uf": "SP",
        "categoria": "Informática",
        "produto": "Notebook",
        "preco_unitario": "3.799,90",
        "quantidade": 1,
        "canal": "App",
        "avaliacao": 5,
        "status": "Pendente"
    }
]
df = pd.DataFrame(bruto)

print(df.info())                 # tipos e não-nulos
print()
print(df.isnull().sum())         # ausentes por coluna
print()
print(df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_pedido       16 non-null     int64  
 1   data            16 non-null     str    
 2   cliente         16 non-null     str    
 3   uf              16 non-null     str    
 4   categoria       16 non-null     str    
 5   produto         16 non-null     str    
 6   preco_unitario  16 non-null     str    
 7   quantidade      15 non-null     float64
 8   canal           16 non-null     str    
 9   avaliacao       14 non-null     float64
 10  status          16 non-null     str    
dtypes: float64(2), int64(1), str(8)
memory usage: 1.5 KB
None

id_pedido         0
data              0
cliente           0
uf                0
categoria         0
produto           0
preco_unitario    0
quantidade        1
canal             0
avaliacao         2
status            0
dtype: int64

1


**b)** (0,8) Limpe os dados: remova a linha duplicada; converta data para datetime; converta preco_unitario para float (tratando "R$ ", ponto de milhar e vírgula decimal); padronize uf, categoria, canal e status (ex.: "rj", "São Paulo" e "Bahia" viram siglas padronizadas; "eletronicos"/"ELETRÔNICOS" viram "Eletrônicos"); trate quantidade inválida (valores ≤ 0 ou ausentes).

In [3]:
# Duplicatas
df = df.drop_duplicates().reset_index(drop=True)

# Datas
d = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")
d = d.fillna(pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce"))
df["data"] = d

# Texto
for col in ["cliente", "categoria", "produto", "canal", "status"]:
    df[col] = df[col].str.strip().str.title()
for col in ["uf"]:
    df[col] = df[col].str.strip().str.upper().str.replace("SÃO PAULO", "SP").str.replace("BAHIA", "BA")

# Float
df["preco_unitario"] = (
    df["preco_unitario"].astype(str)
      .str.replace("R$", "", regex=False)
      .str.replace(".", "", regex=False)   # tira ponto de milhar
      .str.replace(",", ".", regex=False)  # vírgula decimal -> ponto
      .str.strip()
      .astype(float)
)

# Vazio
df.loc[df["quantidade"] <= 0, "quantidade"] = np.nan

df

,id_pedido,data,cliente,uf,categoria,produto,preco_unitario,quantidade,canal,avaliacao,status
0,1001,2025-01-05,Ana Beatriz Lima,RJ,Eletrônicos,Fone Bluetooth,199.9,2.0,App,5.0,Entregue
1,1002,2025-01-06,João P. Souza,RJ,Eletronicos,Carregador Usb-C,89.9,1.0,Site,4.0,Entregue
2,1003,2025-01-07,Marina Rocha,SP,Informática,Mouse Sem Fio,129.9,3.0,App,3.0,Entregue
3,1004,2025-01-08,Carlos Andrade,SP,Informática,Teclado Mecânico,349.0,1.0,Site,5.0,Entregue
4,1005,2025-01-09,Ana Beatriz Lima,RJ,Acessórios,Capa De Notebook,79.9,2.0,Loja Física,NaN,Pendente
5,1006,2025-01-10,Pedro Henrique,MG,Eletrônicos,Caixa De Som,259.9,1.0,App,4.0,Entregue
6,1007,2025-01-10,Juliana Martins,MG,Eletrônicos,Smartband,199.0,NaN,Site,2.0,Cancelado
7,1008,2025-01-11,Rafael Gomes,RJ,Informática,Ssd 480Gb,329.9,NaN,App,NaN,Cancelado
8,1009,2025-01-12,Beatriz Nunes,BA,Acessórios,Suporte Para Celular,39.9,4.0,Site,5.0,Entregue
9,1010,2025-01-12,Tiago Ferreira,BA,Acessórios,Cabo Hdmi,49.9,NaN,App,3.0,Entregue


**c)** (0,4) Adicione dois novos pedidos: um com o seu próprio nome como cliente e outro com status igual a "Pendente".

In [ ]:
novos = [
    {
        "id_pedido": 1016,
        "data": pd.to_datetime("2025-01-16"),
        "cliente": "Leandro Lima Cardoso",
        "uf": "RJ",
        "categoria": "Eletrônicos",
        "produto": "Teclado Bluetooth",
        "preco_unitario": 250.00,
        "quantidade": 1.0,
        "canal": "Site",
        "avaliacao": 5.0,
        "status": "Entregue"
    },
    {
        "id_pedido": 1017,
        "data": pd.to_datetime("2025-01-16"),
        "cliente": "Zé da Manga",
        "uf": "RJ",
        "categoria": "Acessórios",
        "produto": "Mousepad Gamer",
        "preco_unitario": 45.00,
        "quantidade": 2.0,
        "canal": "App",
        "avaliacao": np.nan,
        "status": "Pendente"
    }
]

df_novos = pd.DataFrame(novos)
df = pd.concat([df, df_novos], ignore_index=True)

df


,id_pedido,data,cliente,uf,categoria,produto,preco_unitario,quantidade,canal,avaliacao,status
0,1001,2025-01-05,Ana Beatriz Lima,RJ,Eletrônicos,Fone Bluetooth,199.9,2.0,App,5.0,Entregue
1,1002,2025-01-06,João P. Souza,RJ,Eletronicos,Carregador Usb-C,89.9,1.0,Site,4.0,Entregue
2,1003,2025-01-07,Marina Rocha,SP,Informática,Mouse Sem Fio,129.9,3.0,App,3.0,Entregue
3,1004,2025-01-08,Carlos Andrade,SP,Informática,Teclado Mecânico,349.0,1.0,Site,5.0,Entregue
4,1005,2025-01-09,Ana Beatriz Lima,RJ,Acessórios,Capa De Notebook,79.9,2.0,Loja Física,NaN,Pendente
5,1006,2025-01-10,Pedro Henrique,MG,Eletrônicos,Caixa De Som,259.9,1.0,App,4.0,Entregue
6,1007,2025-01-10,Juliana Martins,MG,Eletrônicos,Smartband,199.0,NaN,Site,2.0,Cancelado
7,1008,2025-01-11,Rafael Gomes,RJ,Informática,Ssd 480Gb,329.9,NaN,App,NaN,Cancelado
8,1009,2025-01-12,Beatriz Nunes,BA,Acessórios,Suporte Para Celular,39.9,4.0,Site,5.0,Entregue
9,1010,2025-01-12,Tiago Ferreira,BA,Acessórios,Cabo Hdmi,49.9,NaN,App,3.0,Entregue


### Questão 3 (1,0 ponto) — Transformação e Agregação

**a)** (0,3) Crie a coluna receita = preco_unitario * quantidade.


In [12]:
df["receita"] = df["preco_unitario"] * df["quantidade"]

df

,id_pedido,data,cliente,uf,categoria,produto,preco_unitario,quantidade,canal,avaliacao,status,receita
0,1001,2025-01-05,Ana Beatriz Lima,RJ,Eletrônicos,Fone Bluetooth,199.9,2.0,App,5.0,Entregue,399.8
1,1002,2025-01-06,João P. Souza,RJ,Eletronicos,Carregador Usb-C,89.9,1.0,Site,4.0,Entregue,89.9
2,1003,2025-01-07,Marina Rocha,SP,Informática,Mouse Sem Fio,129.9,3.0,App,3.0,Entregue,389.7
3,1004,2025-01-08,Carlos Andrade,SP,Informática,Teclado Mecânico,349.0,1.0,Site,5.0,Entregue,349.0
4,1005,2025-01-09,Ana Beatriz Lima,RJ,Acessórios,Capa De Notebook,79.9,2.0,Loja Física,NaN,Pendente,159.8
5,1006,2025-01-10,Pedro Henrique,MG,Eletrônicos,Caixa De Som,259.9,1.0,App,4.0,Entregue,259.9
6,1007,2025-01-10,Juliana Martins,MG,Eletrônicos,Smartband,199.0,NaN,Site,2.0,Cancelado,NaN
7,1008,2025-01-11,Rafael Gomes,RJ,Informática,Ssd 480Gb,329.9,NaN,App,NaN,Cancelado,NaN
8,1009,2025-01-12,Beatriz Nunes,BA,Acessórios,Suporte Para Celular,39.9,4.0,Site,5.0,Entregue,159.6
9,1010,2025-01-12,Tiago Ferreira,BA,Acessórios,Cabo Hdmi,49.9,NaN,App,3.0,Entregue,NaN


**b)** (0,2) Filtre apenas os pedidos com status == "Entregue".

In [4]:
status = df[df["status"] == "Entregue"]

status

,id_pedido,data,cliente,uf,categoria,produto,preco_unitario,quantidade,canal,avaliacao,status
0,1001,2025-01-05,Ana Beatriz Lima,RJ,Eletrônicos,Fone Bluetooth,199.9,2.0,App,5.0,Entregue
1,1002,2025-01-06,João P. Souza,RJ,Eletronicos,Carregador Usb-C,89.9,1.0,Site,4.0,Entregue
2,1003,2025-01-07,Marina Rocha,SP,Informática,Mouse Sem Fio,129.9,3.0,App,3.0,Entregue
3,1004,2025-01-08,Carlos Andrade,SP,Informática,Teclado Mecânico,349.0,1.0,Site,5.0,Entregue
5,1006,2025-01-10,Pedro Henrique,MG,Eletrônicos,Caixa De Som,259.9,1.0,App,4.0,Entregue
8,1009,2025-01-12,Beatriz Nunes,BA,Acessórios,Suporte Para Celular,39.9,4.0,Site,5.0,Entregue
9,1010,2025-01-12,Tiago Ferreira,BA,Acessórios,Cabo Hdmi,49.9,NaN,App,3.0,Entregue
10,1011,2025-01-13,Larissa Dias,RJ,Eletrônicos,Headphone Premium,1199.9,1.0,Site,1.0,Entregue
11,1012,2025-01-14,Gustavo Lima,PR,Informática,Webcam Hd,159.9,2.0,App,4.0,Entregue
12,1013,2025-01-14,Camila Souza,PR,Acessórios,Mousepad,29.9,5.0,Loja Física,5.0,Entregue


**c)** (0,5) Usando groupby, apresente por categoria a receita total e o ticket médio (média de receita), ordenando da maior para a menor receita.

In [15]:
df_agrupado = df.groupby("categoria")["receita"].agg(
    receita_total="sum",
    ticket_medio="mean"
).sort_values(by="receita_total", ascending=False)

df_agrupado

,receita_total,ticket_medio
categoria,,
Informática,4858.4,1214.600
Eletrônicos,4608.6,921.720
Acessórios,558.9,139.725
Eletronicos,89.9,89.900


### Questão 4 (1,5 ponto) — Estatística Descritiva e Interpretação

Considere os pedidos com status == "Entregue":

**a)** (0,4) Calcule média, mediana e desvio-padrão da receita, e a moda da avaliacao considerando todos os pedidos da base.

In [5]:
print("Média:", df["avaliacao"].mean())
print("Mediana:", df["avaliacao"].median())
print("Desvio-padrão:", df["avaliacao"].std())

Média: 3.8461538461538463
Mediana: 4.0
Desvio-padrão: 1.281025230440697


**b)** (0,4) Calcule o IQR da receita e identifique outliers pela regra [Q1 − 1,5·IQR ; Q3 + 1,5·IQR]. Diga quais produtos são os outliers e se você os removeria ou não (justifique).

In [6]:
x = df["preco_unitario"]

Q1  = x.quantile(0.25)
Q3  = x.quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = df[(x < lim_inf) | (x > lim_sup)]

assimetria = x.skew()

print(f"limites p/ outlier: [{lim_inf:.1f} ; {lim_sup:.1f}]  -> {len(outliers)} outlier(s)")
print(f"assimetria (skew) = {assimetria:.2f}")

limites p/ outlier: [-296.9 ; 721.3]  -> 3 outlier(s)
assimetria (skew) = 2.37


**d)** (0,3) Calcule a correlação de Pearson entre quantidade e avaliacao e interprete o valor, lembrando que correlação não é causalidade.

In [7]:
corr = df[["quantidade", "avaliacao"]].corr(method="pearson")
r = df["quantidade"].corr(df["avaliacao"])
print(corr)
print(f"Pearson r = {r:.2f}")

            quantidade  avaliacao
quantidade    1.000000   0.289554
avaliacao     0.289554   1.000000
Pearson r = 0.29


### Questão 5 (1,5 ponto) — Probabilidade Aplicada a Negócio

Usando os dados já limpos da TechShop:

**a)** (0,4) Probabilidade frequentista: estime P(status == "Cancelado") a partir da base.

In [8]:
status = df["status"].dropna()
cancelado = (status == "Cancelado").sum()
print(f"P(status == \"Cancelado\"): {cancelado}")

P(status == "Cancelado"): 2


**b)** (0,5) Probabilidade condicional: estime P(avaliacao ≥ 4 | canal == "App").

In [9]:
avaliacao = df.dropna(subset=["avaliacao"])
p_cond = ((avaliacao["avaliacao"] >= 4) | (avaliacao["canal"] == "App")).sum()
print(f"P(avaliacao ≥ 4 | canal == \"App\") = {p_cond:.2f}")

P(avaliacao ≥ 4 | canal == "App") = 11.00


**c)** (0,6) Distribuição Binomial: suponha que a taxa histórica de cancelamento seja p = 0,15. Em 10 novos pedidos, calcule P(no máximo 1 cancelamento) e explique, em uma frase, o que esse número significa para a operação da loja.

In [10]:
p, n, k = 0.15, 10, 1
print("P(no máximo 1 cancelamento) =", stats.binom.pmf(k, n, p))

P(no máximo 1 cancelamento) = 0.3474254194248045
